# EU Market Pulse — Supabase Data Preparation

## Objective

This notebook prepares the final tables to be loaded into the Supabase 
(Postgres) database:

1. `countries` — reference/dimension table (country code, name, region group)
2. `market_indicators` — fact table (inflation, price level, house price index by country-year)
3. `country_kpis` — aggregated business KPIs by country

Each table is exported as a clean CSV, ready for import via the Supabase 
Table Editor or a direct SQLAlchemy connection.

## 1. Build the Countries Reference Table

We define the ISO-2 country code and region group for each of the 19 
countries in scope. This table will serve as the dimension table for joins 
in SQL and for map visualizations in the Streamlit dashboard.

In [2]:
import pandas as pd

countries_reference = pd.DataFrame([
    {'country_name': 'United Kingdom', 'country_code': 'GB', 'region_group': 'Anchor Market'},
    {'country_name': 'Switzerland',    'country_code': 'CH', 'region_group': 'Anchor Market'},
    {'country_name': 'Ireland',        'country_code': 'IE', 'region_group': 'Anchor Market'},
    {'country_name': 'Germany',        'country_code': 'DE', 'region_group': 'Western Europe'},
    {'country_name': 'France',         'country_code': 'FR', 'region_group': 'Western Europe'},
    {'country_name': 'Netherlands',    'country_code': 'NL', 'region_group': 'Western Europe'},
    {'country_name': 'Austria',        'country_code': 'AT', 'region_group': 'Western Europe'},
    {'country_name': 'Belgium',        'country_code': 'BE', 'region_group': 'Western Europe'},
    {'country_name': 'Portugal',       'country_code': 'PT', 'region_group': 'Southern Europe'},
    {'country_name': 'Spain',          'country_code': 'ES', 'region_group': 'Southern Europe'},
    {'country_name': 'Italy',          'country_code': 'IT', 'region_group': 'Southern Europe'},
    {'country_name': 'Greece',         'country_code': 'GR', 'region_group': 'Southern Europe'},
    {'country_name': 'Poland',         'country_code': 'PL', 'region_group': 'Eastern Europe'},
    {'country_name': 'Romania',        'country_code': 'RO', 'region_group': 'Eastern Europe'},
    {'country_name': 'Hungary',        'country_code': 'HU', 'region_group': 'Eastern Europe'},
    {'country_name': 'Czechia',        'country_code': 'CZ', 'region_group': 'Eastern Europe'},
    {'country_name': 'Sweden',         'country_code': 'SE', 'region_group': 'Nordics'},
    {'country_name': 'Denmark',        'country_code': 'DK', 'region_group': 'Nordics'},
    {'country_name': 'Finland',        'country_code': 'FI', 'region_group': 'Nordics'},
])

print(f"Shape: {countries_reference.shape}")
countries_reference

Shape: (19, 3)


,country_name,country_code,region_group
0,United Kingdom,GB,Anchor Market
1,Switzerland,CH,Anchor Market
2,Ireland,IE,Anchor Market
3,Germany,DE,Western Europe
4,France,FR,Western Europe
5,Netherlands,NL,Western Europe
6,Austria,AT,Western Europe
7,Belgium,BE,Western Europe
8,Portugal,PT,Southern Europe
9,Spain,ES,Southern Europe


## 2. Load Processed Data and Merge with Country Codes

We load the two processed datasets from the EDA notebook and join them with 
`countries_reference` to attach the `country_code`, replacing the full 
country name as the primary linking key going forward.

In [3]:
market_indicators_raw = pd.read_csv('../data/processed/eu_market_pulse_clean.csv')
country_kpis_raw = pd.read_csv('../data/processed/country_kpis.csv')

print(f"market_indicators_raw: {market_indicators_raw.shape}")
print(f"country_kpis_raw: {country_kpis_raw.shape}")
market_indicators_raw.head()

market_indicators_raw: (222, 6)
country_kpis_raw: (18, 8)


,country,year,inflation_rate,price_level_index,house_price_index,house_price_flag
0,Austria,2014,1.5,109.3,NaN,NaN
1,Austria,2015,0.8,109.3,NaN,NaN
2,Austria,2016,1.0,110.4,105.27,NaN
3,Austria,2017,2.2,111.6,108.56,NaN
4,Austria,2018,2.1,112.1,112.62,NaN


## 3. Attach Country Codes and Build Final Tables

We merge both datasets with `countries_reference` to attach `country_code`, 
then build the three final tables matching the Supabase schema:

- `market_indicators` — replaces `country` (name) with `country_code`
- `country_kpis` — same, with `country_code` as the primary key

In [4]:
# Build market_indicators table
market_indicators = market_indicators_raw.merge(
    countries_reference[['country_name', 'country_code']],
    left_on='country', right_on='country_name'
)[['country_code', 'year', 'inflation_rate', 'price_level_index', 'house_price_index']]

# Build country_kpis table
country_kpis = country_kpis_raw.merge(
    countries_reference[['country_name', 'country_code']],
    left_on='country', right_on='country_name'
)[['country_code', 'cost_of_living_score', 'inflation_stability_score', 
   'housing_pressure_score', 'overall_expansion_score']]

print(f"market_indicators: {market_indicators.shape}")
print(f"country_kpis: {country_kpis.shape}")

# Validation: confirm no rows were lost in the merge
print(f"\nRows lost in market_indicators merge: {len(market_indicators_raw) - len(market_indicators)}")
print(f"Rows lost in country_kpis merge: {len(country_kpis_raw) - len(country_kpis)}")

market_indicators.head()

market_indicators: (222, 5)
country_kpis: (18, 5)

Rows lost in market_indicators merge: 0
Rows lost in country_kpis merge: 0


,country_code,year,inflation_rate,price_level_index,house_price_index
0,AT,2014,1.5,109.3,NaN
1,AT,2015,0.8,109.3,NaN
2,AT,2016,1.0,110.4,105.27
3,AT,2017,2.2,111.6,108.56
4,AT,2018,2.1,112.1,112.62


## 4. Export Final Tables

The three tables are exported to `data/processed/` as clean CSVs, ready to 
be imported into Supabase via the Table Editor or a direct database connection.

In [5]:
countries_reference.to_csv('../data/processed/countries.csv', index=False)
market_indicators.to_csv('../data/processed/market_indicators.csv', index=False)
country_kpis.to_csv('../data/processed/country_kpis_final.csv', index=False)

print("Exported:")
print(f"  countries.csv: {countries_reference.shape}")
print(f"  market_indicators.csv: {market_indicators.shape}")
print(f"  country_kpis_final.csv: {country_kpis.shape}")

Exported:
  countries.csv: (19, 3)
  market_indicators.csv: (222, 5)
  country_kpis_final.csv: (18, 5)
